# 🤟 Sign Language Detection System — Unified Colab Training Pipeline

This notebook contains the complete end-to-end pipeline to train, evaluate, and export all three models:
1. **MLP Model** (Static Alphabet/Letter mode)
2. **BiLSTM Model** (Dynamic Gesture/Word mode)
3. **MobileNetV3 Small CNN** (Fallback image crop mode)

**Instructions for Google Colab:**
1. Connect to a GPU runtime (`Runtime` -> `Change runtime type` -> select `T4 GPU` or similar).
2. Upload your `kaggle.json` file to the Colab files area or configure your Kaggle credentials.
3. Run all cells to download the dataset, extract landmarks, train, and export models.


## 1. Environment Setup & Dependencies


In [1]:
# Install required packages in Google Colab environment
!pip install mediapipe albumentations scikit-learn tensorflow matplotlib seaborn tqdm gdown opencv-python-headless


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 88.8 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 10.6 MB/s eta 0:00:00
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
    Uninstalling absl-py-1.4.0:
      Successfully uninstalled absl-py-1.4.0


## 2. Imports, Paths & Global Configuration


In [2]:
import os
import sys
import json
import random
import shutil
import base64
import zipfile
import logging
import threading
import time
from pathlib import Path
from dataclasses import dataclass, field, asdict
from datetime import datetime
from typing import Optional

import cv2
import numpy as np
import pandas as pd
import mediapipe as mp
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelBinarizer, StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import classification_report, confusion_matrix, top_k_accuracy_score

import tensorflow as tf
import keras
from keras import layers

# --- Path Configuration ---
BASE_DIR = Path("/content/project")
BASE_DIR.mkdir(parents=True, exist_ok=True)

RAW_DIR = BASE_DIR / "data" / "raw" / "ASL"
PROCESSED_DIR = BASE_DIR / "data" / "processed" / "ASL"
DOWNLOAD_DIR = BASE_DIR / "data" / "downloads"
MODELS_DIR = BASE_DIR / "models"
LOGS_DIR = BASE_DIR / "logs"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
LOGS_DIR.mkdir(parents=True, exist_ok=True)

# --- Class Registry ---
ASL_CLASSES = [*list("ABCDEFGHIJKLMNOPQRSTUVWXYZ"), "space", "del", "nothing"]
NUM_CLASSES = len(ASL_CLASSES)
FEATURE_DIM = 63  # 21 landmarks * 3 coordinates

print(f"Project directory set to: {BASE_DIR}")
print(f"Dialect classes registered: {ASL_CLASSES}")


Project directory set to: /content/project
Dialect classes registered: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'space', 'del', 'nothing']


## 3. Dataset Download (Kaggle ASL Alphabet)


In [3]:
# Setup Kaggle credentials
# Upload your kaggle.json file to Colab's default /content/ folder before running this cell.
kaggle_json_colab = Path("C:\projects\Sign Language Detection System\kaggle.json")
kaggle_json_dest = Path("C:\projects\Sign Language Detection System\kaggle.json")

if kaggle_json_colab.exists():
    kaggle_json_dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(str(kaggle_json_colab), str(kaggle_json_dest))
    os.chmod(str(kaggle_json_dest), 0o600)
    print("✅ Kaggle API Token configured successfully.")
else:
    print("⚠️  Warning: kaggle.json not found in /content/.")
    print("Please upload your API key to Colab, or manually download the dataset to /content/project/data/downloads/extracted.")

# Download dataset via Kaggle CLI
zip_path = DOWNLOAD_DIR / "asl-alphabet.zip"
if not zip_path.exists():
    print("Downloading asl-alphabet dataset (~1GB)...")
    !kaggle datasets download -d grassknoted/asl-alphabet -p {DOWNLOAD_DIR}
else:
    print("Dataset already downloaded.")

# Extract & organize
extract_dir = DOWNLOAD_DIR / "extracted"
if not extract_dir.exists():
    print("Extracting zip dataset...")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(str(extract_dir))
    print("Extraction complete.")
else:
    print("Dataset already extracted.")

# Move / Organize into class structure
print("Organizing dataset...")
train_root = extract_dir / "asl_alphabet_train" / "asl_alphabet_train"
if not train_root.exists():
    train_root = extract_dir / "asl_alphabet_train"
if not train_root.exists():
    train_root = extract_dir

moved_total = 0
for class_label in ASL_CLASSES:
    src_dir = train_root / class_label
    dst_dir = RAW_DIR / class_label
    dst_dir.mkdir(parents=True, exist_ok=True)

    if src_dir.exists():
        images = list(src_dir.glob("*.jpg"))
        existing = len(list(dst_dir.glob("*.jpg")))
        if existing < len(images):
            # Limit samples to 300 per class for faster training/processing in this walkthrough if desired
            # Otherwise copy all by removing slicing
            for img_path in images[:400]:
                shutil.copy2(str(img_path), str(dst_dir / img_path.name))
                moved_total += 1
            print(f"  Organized {class_label} -> copied {len(images[:400])} images")
        else:
            print(f"  {class_label} already organized.")

print(f"Dataset organization complete. Organized samples: {moved_total}")


<>:3: SyntaxWarning: invalid escape sequence '\p'
<>:4: SyntaxWarning: invalid escape sequence '\p'
<>:3: SyntaxWarning: invalid escape sequence '\p'
<>:4: SyntaxWarning: invalid escape sequence '\p'
/tmp/ipykernel_818/3206298814.py:3: SyntaxWarning: invalid escape sequence '\p'
  kaggle_json_colab = Path("C:\projects\Sign Language Detection System\kaggle.json")
/tmp/ipykernel_818/3206298814.py:4: SyntaxWarning: invalid escape sequence '\p'
  kaggle_json_dest = Path("C:\projects\Sign Language Detection System\kaggle.json")


⚠️  Warning: kaggle.json not found in /content/.
Please upload your API key to Colab, or manually download the dataset to /content/project/data/downloads/extracted.
Dataset URL: https://www.kaggle.com/datasets/grassknoted/asl-alphabet
License(s): GPL-2.0
100% 1.03G/1.03G [00:08<00:00, 125MB/s]

Extracting zip dataset...
Extraction complete.
Organizing dataset...
  Organized A -> copied 400 images
  Organized B -> copied 400 images
  Organized C -> copied 400 images
  Organized D -> copied 400 images
  Organized E -> copied 400 images
  Organized F -> copied 400 images
  Organized G -> copied 400 images
  Organized H -> copied 400 images
  Organized I -> copied 400 images
  Organized J -> copied 400 images
  Organized K -> copied 400 images
  Organized L -> copied 400 images
  Organized M -> copied 400 images
  Organized N -> copied 400 images
  Organized O -> copied 400 images
  Organized P -> copied 400 images
  Organized Q -> copied 400 images
  Organized R -> copied 400 images
  Org

## 4. MediaPipe Landmark Extraction & Preprocessing


In [7]:
# Force reinstall a compatible version and restart
!pip install mediapipe==0.10.9 --quiet

ERROR: Could not find a version that satisfies the requirement mediapipe==0.10.9 (from versions: 0.10.13, 0.10.14, 0.10.15, 0.10.18, 0.10.20, 0.10.21, 0.10.30, 0.10.31, 0.10.32, 0.10.33, 0.10.35)
ERROR: No matching distribution found for mediapipe==0.10.9


In [8]:
# Preprocessing configurations
@dataclass
class PreprocessConfig:
    class_label: Optional[str] = None
    augment: bool = True
    aug_factor: int = 3
    crop_size: int = 224
    padding_ratio: float = 0.2
    output_landmarks: bool = True
    output_crops: bool = True

cfg = PreprocessConfig()




# Setup MediaPipe
# Lazy-import mediapipe to avoid "import not at top" lint and handle missing package
import importlib
import sys

# Remove cached mediapipe if already imported
if 'mediapipe' in sys.modules:
    del sys.modules['mediapipe']

import mediapipe as mp
mp_hands = mp.solutions.hands

def extract_landmarks(image_bgr):
    rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    results = hands.process(rgb)
    if not results.multi_hand_landmarks:
        return None
    hand_lm = results.multi_hand_landmarks[0]
    landmarks = np.array([[lm.x, lm.y, lm.z] for lm in hand_lm.landmark], dtype=np.float32)
    return landmarks.flatten()

def normalize_landmarks(raw):
    landmarks = raw.reshape(21, 3)
    wrist = landmarks[0].copy()
    landmarks = landmarks - wrist
    
    xy = landmarks[:, :2]
    bbox_min = xy.min(axis=0)
    bbox_max = xy.max(axis=0)
    bbox_size = bbox_max - bbox_min
    diagonal = float(np.linalg.norm(bbox_size))
    
    if diagonal < 1e-6:
        return np.zeros(63, dtype=np.float32)
        
    landmarks = landmarks / diagonal
    return landmarks.flatten().astype(np.float32)

def extract_hand_crop(image_bgr, hands_result, crop_size=224, padding_ratio=0.2):
    if not hands_result.multi_hand_landmarks:
        return None
    h, w = image_bgr.shape[:2]
    hand_lm = hands_result.multi_hand_landmarks[0]
    xs = [lm.x * w for lm in hand_lm.landmark]
    ys = [lm.y * h for lm in hand_lm.landmark]
    x_min, x_max = int(min(xs)), int(max(xs))
    y_min, y_max = int(min(ys)), int(max(ys))
    
    pad_x = int((x_max - x_min) * padding_ratio)
    pad_y = int((y_max - y_min) * padding_ratio)
    x1, y1 = max(0, x_min - pad_x), max(0, y_min - pad_y)
    x2, y2 = min(w, x_max + pad_x), min(h, y_max + pad_y)
    
    if x2 <= x1 or y2 <= y1:
        return None
    crop = image_bgr[y1:y2, x1:x2]
    return cv2.resize(crop, (crop_size, crop_size), interpolation=cv2.INTER_LANCZOS4)

def augment_landmarks(landmarks):
    augmented = []
    lm = landmarks.reshape(21, 3)
    
    # 1. Mirror
    flipped = lm.copy()
    flipped[:, 0] = -flipped[:, 0]
    augmented.append(flipped.flatten())
    
    # 2. Gaussian Jitter
    jitter = lm + np.random.randn(*lm.shape).astype(np.float32) * 0.02
    augmented.append(jitter.flatten())
    
    # 3. Scale
    scale = random.uniform(0.85, 1.15)
    scaled = lm * scale
    augmented.append(scaled.flatten())
    
    return augmented

# Run Processing
all_landmarks = []
all_labels = []
crops_dir = PROCESSED_DIR / "crops"
crops_dir.mkdir(parents=True, exist_ok=True)

print("Extracting landmarks and generating hand crops...")
for class_index, class_label in enumerate(tqdm(ASL_CLASSES)):
    raw_images = sorted((RAW_DIR / class_label).glob("*.jpg"))
    skipped = 0
    for img_path in raw_images:
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        raw_lm = extract_landmarks(img)
        if raw_lm is None:
            skipped += 1
            continue
        norm_lm = normalize_landmarks(raw_lm)
        all_landmarks.append(norm_lm)
        all_labels.append(class_index)
        
        # Save image crop for CNN
        if cfg.output_crops:
            rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            crop_result = hands.process(rgb)
            crop = extract_hand_crop(img, crop_result, cfg.crop_size, cfg.padding_ratio)
            if crop is not None:
                crop_label_dir = crops_dir / class_label
                crop_label_dir.mkdir(exist_ok=True)
                cv2.imwrite(str(crop_label_dir / img_path.name), crop)
                
        # Landmark augmentation
        if cfg.augment:
            aug_lms = augment_landmarks(norm_lm)
            for aug in aug_lms[:cfg.aug_factor]:
                all_landmarks.append(aug)
                all_labels.append(class_index)

    print(f"  Class {class_label}: processed={len(raw_images)-skipped}, skipped={skipped}")

hands.close()

X = np.array(all_landmarks, dtype=np.float32)
y = np.array(all_labels, dtype=np.int32)
np.save(str(PROCESSED_DIR / "landmarks_all.npy"), X)
np.save(str(PROCESSED_DIR / "labels_all.npy"), y)
print(f"✅ Landmark extraction completed! Features: {X.shape}, Labels: {y.shape}")


AttributeError: module 'mediapipe' has no attribute 'solutions'

## 5. Stratified Dataset Splitting


In [ ]:
X = np.load(str(PROCESSED_DIR / "landmarks_all.npy"))
y = np.load(str(PROCESSED_DIR / "labels_all.npy"))

# Split: 70% Train, 15% Val, 15% Test
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.1764, random_state=42, stratify=y_trainval  # 0.15 / 0.85 ≈ 0.1764
)

splits = {
    "X_train": X_train, "y_train": y_train,
    "X_val": X_val, "y_val": y_val,
    "X_test": X_test, "y_test": y_test
}
for name, arr in splits.items():
    np.save(str(PROCESSED_DIR / f"{name}.npy"), arr)
    print(f"Saved {name:<10} shape={arr.shape}")

# Save split metadata
metadata = {
    "train_samples": len(X_train),
    "val_samples": len(X_val),
    "test_samples": len(X_test),
    "num_classes": len(ASL_CLASSES)
}
with open(PROCESSED_DIR / "split_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)


## 6. Exploratory Data Analysis (EDA)


In [ ]:
# Let's plot the distribution of classes
counts = [len(list((RAW_DIR / label).glob("*.jpg"))) for label in ASL_CLASSES]

plt.style.use("dark_background")
fig, ax = plt.subplots(figsize=(16, 5))
ax.bar(ASL_CLASSES, counts, color=plt.cm.Set2(np.linspace(0, 1, 29)), alpha=0.8)
ax.set_title("ASL Dataset — Class Distribution", fontsize=16, pad=15)
ax.set_xlabel("Class")
ax.set_ylabel("Count")
plt.show()

# Run PCA to inspect landmark separation
X_scaled = StandardScaler().fit_transform(X[:4000])
X_pca = PCA(n_components=2, random_state=42).fit_transform(X_scaled)

fig, ax = plt.subplots(figsize=(12, 8))
cmap = plt.cm.get_cmap("tab20", len(ASL_CLASSES))
for i, label in enumerate(ASL_CLASSES):
    mask = y[:4000] == i
    if mask.sum() > 0:
        ax.scatter(X_pca[mask, 0], X_pca[mask, 1], c=[cmap(i)], label=label, alpha=0.6, s=15)
ax.set_title("PCA 2D Projection of landmark space")
ax.legend(ncol=3, fontsize=8)
plt.show()


## 7. Hyperparameter Configs & Data Generators


In [ ]:
# Hyperparameter Configurations
@dataclass
class MLPConfig:
    input_dim: int = FEATURE_DIM
    hidden_dims: list = field(default_factory=lambda: [512, 256, 128])
    num_classes: int = NUM_CLASSES
    dropout_rate: float = 0.4
    use_batch_norm: bool = True
    epochs: int = 60
    batch_size: int = 64
    learning_rate: float = 1e-3
    lr_decay_factor: float = 0.5
    lr_patience: int = 6
    early_stop_patience: int = 12
    label_smoothing: float = 0.1
    log_dir: Path = LOGS_DIR / "mlp"
    save_dir: Path = MODELS_DIR

@dataclass
class LSTMConfig:
    input_dim: int = FEATURE_DIM
    sequence_len: int = 30
    lstm_units: list = field(default_factory=lambda: [128, 64])
    num_classes: int = NUM_CLASSES
    dropout_rate: float = 0.3
    bidirectional: bool = True
    epochs: int = 40
    batch_size: int = 32
    learning_rate: float = 5e-4
    lr_decay_factor: float = 0.5
    lr_patience: int = 5
    early_stop_patience: int = 10
    label_smoothing: float = 0.05
    gradient_clip: float = 1.0
    log_dir: Path = LOGS_DIR / "lstm"
    save_dir: Path = MODELS_DIR

@dataclass
class CNNConfig:
    input_shape: tuple = (224, 224, 3)
    num_classes: int = NUM_CLASSES
    base_model: str = "MobileNetV3Small"
    fine_tune_from: int = 80
    dropout_rate: float = 0.3
    phase1_epochs: int = 15
    phase1_lr: float = 1e-3
    phase1_batch_size: int = 32
    phase2_epochs: int = 25
    phase2_lr: float = 1e-5
    phase2_batch_size: int = 16
    label_smoothing: float = 0.1
    early_stop_patience: int = 8
    log_dir: Path = LOGS_DIR / "cnn"
    save_dir: Path = MODELS_DIR

# --- Load Datasets ---
def load_splits():
    X_tr = np.load(str(PROCESSED_DIR / "X_train.npy"))
    y_tr = np.load(str(PROCESSED_DIR / "y_train.npy"))
    X_val = np.load(str(PROCESSED_DIR / "X_val.npy"))
    y_val = np.load(str(PROCESSED_DIR / "y_val.npy"))
    X_test = np.load(str(PROCESSED_DIR / "X_test.npy"))
    y_test = np.load(str(PROCESSED_DIR / "y_test.npy"))

    lb = LabelBinarizer()
    y_tr_ohe = lb.fit_transform(y_tr)
    y_val_ohe = lb.transform(y_val)
    y_test_ohe = lb.transform(y_test)

    return X_tr, X_val, X_test, y_tr_ohe, y_val_ohe, y_test_ohe

# --- sequence generator for LSTM ---
def build_sequence_dataset(X, y, seq_len, batch_size, shuffle=True):
    N, feature_dim = len(X), X.shape[1]
    num_classes = y.shape[1]

    def generator():
        indices = np.random.permutation(N) if shuffle else np.arange(N)
        for i in indices:
            base = X[i]
            # add small random jittering per sequence step to simulate hand motion
            seq = np.stack([
                base + np.random.randn(feature_dim).astype(np.float32) * 0.005
                for _ in range(seq_len)
            ], axis=0)
            yield seq, y[i]

    ds = tf.data.Dataset.from_generator(
        generator,
        output_signature=(
            tf.TensorSpec(shape=(seq_len, feature_dim), dtype=tf.float32),
            tf.TensorSpec(shape=(num_classes,), dtype=tf.float32),
        ),
    )
    if shuffle:
        ds = ds.shuffle(buffer_size=min(N, 2000))
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

# --- Image loading generator for CNN ---
def build_image_dataset(split, cfg):
    crops_dir = PROCESSED_DIR / "crops"
    class_names = sorted(d.name for d in crops_dir.iterdir() if d.is_dir())
    subset_map = {"train": "training", "val": "validation"}
    keras_subset = subset_map.get(split, split)
    use_split = keras_subset in ("training", "validation")

    return tf.keras.utils.image_dataset_from_directory(
        crops_dir,
        labels="inferred",
        label_mode="categorical",
        class_names=class_names,
        image_size=(cfg.input_shape[0], cfg.input_shape[1]),
        batch_size=cfg.phase1_batch_size if split=="train" else cfg.phase2_batch_size,
        shuffle=(split == "train"),
        seed=42,
        validation_split=0.2 if use_split else None,
        subset=keras_subset if use_split else None,
    ).prefetch(tf.data.AUTOTUNE)

def get_callbacks(log_dir, checkpoint_path, patience):
    log_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
    return [
        keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=patience, restore_best_weights=True),
        keras.callbacks.ModelCheckpoint(filepath=str(checkpoint_path), monitor="val_accuracy", save_best_only=True),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=patience//2)
    ]


## 8. Model Architectures & Compilers


In [ ]:
def build_mlp(cfg: MLPConfig) -> keras.Model:
    inp = keras.Input(shape=(cfg.input_dim,), name="landmarks")
    x = inp
    for i, units in enumerate(cfg.hidden_dims):
        x = layers.Dense(units, name=f"dense_{i}")(x)
        if cfg.use_batch_norm:
            x = layers.BatchNormalization(name=f"bn_{i}")(x)
        x = layers.Activation("relu", name=f"relu_{i}")(x)
        x = layers.Dropout(cfg.dropout_rate, name=f"drop_{i}")(x)

    x = layers.Dense(cfg.num_classes, name="logits")(x)
    out = layers.Softmax(name="predictions")(x)
    return keras.Model(inputs=inp, outputs=out, name="ASL_MLP")

def build_lstm(cfg: LSTMConfig) -> keras.Model:
    inp = keras.Input(shape=(cfg.sequence_len, cfg.input_dim), name="landmark_sequence")
    x = inp
    for i, units in enumerate(cfg.lstm_units):
        lstm = layers.LSTM(
            units,
            return_sequences=True,
            dropout=cfg.dropout_rate,
            recurrent_dropout=0.0,
            name=f"lstm_{i}",
        )
        if cfg.bidirectional:
            x = layers.Bidirectional(lstm, name=f"bilstm_{i}")(x)
        else:
            x = lstm(x)
        x = layers.Dropout(cfg.dropout_rate, name=f"drop_{i}")(x)

    x = layers.GlobalAveragePooling1D(name="gap")(x)
    x = layers.Dense(cfg.num_classes, name="logits")(x)
    out = layers.Softmax(name="predictions")(x)
    return keras.Model(inputs=inp, outputs=out, name="ASL_LSTM")

def build_cnn(cfg: CNNConfig) -> keras.Model:
    base = tf.keras.applications.MobileNetV3Small(
        input_shape=cfg.input_shape,
        include_top=False,
        weights="imagenet",
        pooling="avg",
    )
    base.trainable = False

    inp = keras.Input(shape=cfg.input_shape, name="hand_crop")
    x = tf.keras.applications.mobilenet_v3.preprocess_input(inp)
    x = base(x, training=False)
    x = layers.Dropout(cfg.dropout_rate, name="drop_head")(x)
    x = layers.Dense(cfg.num_classes, name="logits")(x)
    out = layers.Softmax(name="predictions")(x)
    return keras.Model(inputs=inp, outputs=out, name="ASL_MobileNetV3")

def compile_model(model, lr, label_smoothing=0.0, clipnorm=None):
    opt_args = {"learning_rate": lr}
    if clipnorm:
        opt_args["clipnorm"] = clipnorm
    model.compile(
        optimizer=keras.optimizers.Adam(**opt_args),
        loss=keras.losses.CategoricalCrossentropy(label_smoothing=label_smoothing),
        metrics=["accuracy", keras.metrics.TopKCategoricalAccuracy(k=5, name="top5")]
    )


## 9. Train Model 1: MLP (Alphabet Classifier)


In [ ]:
mlp_cfg = MLPConfig()
X_tr, X_val, X_test, y_tr_ohe, y_val_ohe, y_test_ohe = load_splits()

mlp_model = build_mlp(mlp_cfg)
compile_model(mlp_model, mlp_cfg.learning_rate, mlp_cfg.label_smoothing)
mlp_model.summary()

print("Training MLP landmark classifier...")
mlp_model.fit(
    X_tr, y_tr_ohe,
    validation_data=(X_val, y_val_ohe),
    epochs=mlp_cfg.epochs,
    batch_size=mlp_cfg.batch_size,
    callbacks=get_callbacks(mlp_cfg.log_dir, mlp_cfg.save_dir / "asl_mlp_best.keras", mlp_cfg.early_stop_payout if hasattr(mlp_cfg, "early_stop_payout") else mlp_cfg.early_stop_patience),
    verbose=1
)
mlp_model.save(str(mlp_cfg.save_dir / "asl_mlp.keras"))
print("✅ MLP trained and saved successfully.")


## 10. Train Model 2: BiLSTM (Sequence Word Classifier)


In [ ]:
lstm_cfg = LSTMConfig()

train_ds = build_sequence_dataset(X_tr, y_tr_ohe, lstm_cfg.sequence_len, lstm_cfg.batch_size)
val_ds = build_sequence_dataset(X_val, y_val_ohe, lstm_cfg.sequence_len, lstm_cfg.batch_size, shuffle=False)
test_ds = build_sequence_dataset(X_test, y_test_ohe, lstm_cfg.sequence_len, lstm_cfg.batch_size, shuffle=False)

lstm_model = build_lstm(lstm_cfg)
compile_model(lstm_model, lstm_cfg.learning_rate, lstm_cfg.label_smoothing, lstm_cfg.gradient_clip)
lstm_model.summary()

print("Training LSTM sequence classifier...")
lstm_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=lstm_cfg.epochs,
    callbacks=get_callbacks(lstm_cfg.log_dir, lstm_cfg.save_dir / "asl_lstm_best.keras", lstm_cfg.early_stop_patience),
    verbose=1
)
lstm_model.save(str(lstm_cfg.save_dir / "asl_lstm.keras"))
print("✅ LSTM trained and saved successfully.")


## 11. Train Model 3: MobileNetV3 CNN (Fallback Image Classifier)


In [ ]:
cnn_cfg = CNNConfig()

# Build datasets
try:
    train_img_ds = build_image_dataset("train", cnn_cfg)
    val_img_ds = build_image_dataset("val", cnn_cfg)
    
    cnn_model = build_cnn(cnn_cfg)
    
    # Phase 1: Train classification head
    print("Training CNN Head (Base Frozen)...")
    compile_model(cnn_model, cnn_cfg.phase1_lr, cnn_cfg.label_smoothing)
    cnn_model.fit(
        train_img_ds,
        validation_data=val_img_ds,
        epochs=cnn_cfg.phase1_epochs,
        callbacks=get_callbacks(cnn_cfg.log_dir / "p1", cnn_cfg.save_dir / "asl_mobilenet_best.keras", cnn_cfg.early_stop_patience),
        verbose=1
    )
    
    # Phase 2: Unfreeze base layers for fine-tuning
    print("Fine-tuning top layers of base CNN...")
    base_cnn = next(layer for layer in cnn_model.layers if isinstance(layer, keras.Model))
    base_cnn.trainable = True
    for layer in base_cnn.layers[:cnn_cfg.fine_tune_from]:
        layer.trainable = False
        
    compile_model(cnn_model, cnn_cfg.phase2_lr, cnn_cfg.label_smoothing)
    cnn_model.fit(
        train_img_ds,
        validation_data=val_img_ds,
        epochs=cnn_cfg.phase2_epochs,
        callbacks=get_callbacks(cnn_cfg.log_dir / "p2", cnn_cfg.save_dir / "asl_mobilenet_best.keras", cnn_cfg.early_stop_patience),
        verbose=1
    )
    
    cnn_model.save(str(cnn_cfg.save_dir / "asl_mobilenet.keras"))
    print("✅ MobileNetV3 CNN trained and saved successfully.")
    
except FileNotFoundError as exc:
    print(f"Skipping CNN training. Image crops directory not found: {exc}")


## 12. Model Evaluations & Reports


In [ ]:
def run_evaluation(model, X_eval, y_eval_ohe, name):
    loss, acc, top5 = model.evaluate(X_eval, y_eval_ohe, verbose=0)
    print("=" * 60)
    print(f"  📊 {name} Performance Summary")
    print(f"  Loss     : {loss:.4f}")
    print(f"  Accuracy : {acc:.4f} ({acc*100:.2f}%)")
    print(f"  Top-5 Acc: {top5:.4f} ({top5*100:.2f}%)")
    print("=" * 60)
    
    # Get classification report
    preds = model.predict(X_eval, verbose=0)
    y_pred = np.argmax(preds, axis=1)
    y_true = np.argmax(y_eval_ohe, axis=1)
    
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, target_names=ASL_CLASSES, zero_division=0))
    
    # Plot normalized confusion matrix
    cm = confusion_matrix(y_true, y_pred, normalize="true")
    fig, ax = plt.subplots(figsize=(14, 12))
    sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues", xticklabels=ASL_CLASSES, yticklabels=ASL_CLASSES, ax=ax, annot_kws={"size": 6})
    ax.set_title(f"{name} Normalized Confusion Matrix")
    plt.tight_layout()
    plt.show()

# Run evaluation on MLP
print("MLP Model Evaluation:")
run_evaluation(mlp_model, X_test, y_test_ohe, "ASL MLP")


## 13. Export Saved Models to Google Drive or Local Downloads


In [ ]:
# Helper cells to download files directly from Google Colab or save to drive
from google.colab import files

print("Run this cell to select which model to download:")
# files.download('/content/project/models/asl_mlp.keras')
# files.download('/content/project/models/asl_lstm.keras')
# files.download('/content/project/models/asl_mobilenet.keras')
